# 03 – The Librarian: Advanced Hybrid RAG

**Operation Ledger-Mind: The Financial Intelligence**

In [ ]:
# Setup
import sys
import os
from pathlib import Path
from dotenv import load_dotenv
import json
from typing import List, Dict, Any

sys.path.append(str(Path.cwd().parent))
load_dotenv()

from src.services.llm_services import (
    load_config,
    get_llm,
    get_text_embeddings
)

config = load_config("../src/config/config.yaml")
llm = get_llm(config)
embeddings = get_text_embeddings(config)
print("LLM and embeddings ready")

In [ ]:
# Load chunks from Notebook 01 (same chunk_ids as the Q&A dataset)
chunks_path = Path(config['artifacts_root']) / "chunks.jsonl"
if not chunks_path.exists():
    raise FileNotFoundError(f"{chunks_path} not found, run Notebook 01 Step 1 first")

with open(chunks_path) as f:
    chunks = [json.loads(line) for line in f]

print(f"Loaded {len(chunks)} chunks")
print(f"   Text chunks: {sum(c['type'] == 'text' for c in chunks)}")
print(f"   Table chunks: {sum(c['type'] == 'table_chunk' for c in chunks)}")

In [ ]:
import weaviate
from weaviate.classes.config import Configure, Property, DataType

client = weaviate.connect_to_embedded(
    version="1.27.3"
)

collection_name = "UberFinancials"

if client.collections.exists(collection_name):
    client.collections.delete(collection_name)

collection = client.collections.create(
    name=collection_name,
    vector_config=Configure.Vectors.self_provided(),
    properties=[
        Property(name="chunk_id", data_type=DataType.INT),
        Property(name="text", data_type=DataType.TEXT),
        Property(name="page", data_type=DataType.INT),
        Property(name="chunk_type", data_type=DataType.TEXT)
    ]
)

print(f"Created collection: {collection_name}")

In [ ]:
# Index chunks
chunk_texts = [c['text'] for c in chunks]
vectors = embeddings.embed_documents(chunk_texts)

collection = client.collections.get(collection_name)

with collection.batch.dynamic() as batch:
    for chunk, vector in zip(chunks, vectors):
        batch.add_object(
            properties={
                "chunk_id": chunk['chunk_id'],
                "text": chunk['text'],
                "page": chunk['page'],
                "chunk_type": chunk['type']
            },
            vector=vector
        )

collection.aggregate.over_all(total_count=True)
print(f"Indexed {len(chunks)} chunks")

In [ ]:
# Search functions
def dense_search(query, top_k=10):
    query_vector = embeddings.embed_query(query)
    collection = client.collections.get(collection_name)
    response = collection.query.near_vector(near_vector=query_vector, limit=top_k, return_metadata=['distance'])
    return [{'chunk_id': obj.properties['chunk_id'], 'text': obj.properties['text'], 'score': 1 - obj.metadata.distance}
            for obj in response.objects]

def bm25_search(query, top_k=10):
    collection = client.collections.get(collection_name)
    response = collection.query.bm25(query=query, limit=top_k)
    return [{'chunk_id': obj.properties['chunk_id'], 'text': obj.properties['text'], 'score': obj.metadata.score or 0.0}
            for obj in response.objects]

print("Search functions ready")

In [ ]:
# Reciprocal Rank Fusion
def reciprocal_rank_fusion(dense_results, bm25_results, k=60):
    rrf_scores = {}

    for rank, result in enumerate(dense_results, 1):
        cid = result['chunk_id']
        if cid not in rrf_scores:
            rrf_scores[cid] = {'text': result['text'], 'chunk_id': cid, 'rrf_score': 0.0}
        rrf_scores[cid]['rrf_score'] += 1 / (k + rank)

    for rank, result in enumerate(bm25_results, 1):
        cid = result['chunk_id']
        if cid not in rrf_scores:
            rrf_scores[cid] = {'text': result['text'], 'chunk_id': cid, 'rrf_score': 0.0}
        rrf_scores[cid]['rrf_score'] += 1 / (k + rank)

    return sorted(rrf_scores.values(), key=lambda x: x['rrf_score'], reverse=True)

print("RRF ready")

In [ ]:
# Cross-encoder reranking
from sentence_transformers import CrossEncoder

reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def rerank(query, candidates, top_k=5):
    pairs = [[query, c['text']] for c in candidates]
    scores = reranker.predict(pairs)

    for candidate, score in zip(candidates, scores):
        candidate['rerank_score'] = float(score)

    return sorted(candidates, key=lambda x: x['rerank_score'], reverse=True)[:top_k]

print("Reranker ready")

In [ ]:
# Complete RAG pipeline
def query_librarian(question, return_sources=False):
    # Hybrid retrieval
    dense = dense_search(question, top_k=10)
    bm25 = bm25_search(question, top_k=10)

    # Fusion
    fused = reciprocal_rank_fusion(dense, bm25)

    # Rerank
    reranked = rerank(question, fused, top_k=5)

    # Build context
    context = "\n\n---\n\n".join([r['text'] for r in reranked])

    # Generate
    prompt = f'''Answer based ONLY on context from Uber's 2024 Annual Report.

    Context:
    {context}

    Question: {question}

    Answer:'''

    response = llm.invoke(prompt)
    answer = response.content

    if return_sources:
        return {'answer': answer, 'sources': reranked, 'context': context}
    return answer

print("query_librarian() function ready")
print("\nTesting...")
print(query_librarian("What is Uber's mission?"))
print("\nThe Librarian is ready!")
client.close()